# LSB eHealth — Medical Track (PhysioNet 2017 ECG)


**Runtime:** 2 arms × 6 seeds × 2 scenarios = 24 runs at roughly 90 s each, so about **35–40 minutes**
on a T4.


In [ ]:
# ============================================================
# 1 — GPU check, config, dataset
# ============================================================
import torch
if not torch.cuda.is_available():
    raise SystemExit('NO GPU. Runtime -> Change runtime type -> T4 GPU, then rerun. '
                     'On CPU each run takes ~40 minutes instead of ~4.')
print('GPU:', torch.cuda.get_device_name(0))

from google.colab import drive; drive.mount('/content/drive', force_remount=False)

from pathlib import Path
from collections import OrderedDict
import json, math, time, random, zipfile, urllib.request
import numpy as np, pandas as pd
import torch.nn as nn, torch.nn.functional as F

DRIVE = Path('/content/drive/MyDrive')
OUT   = DRIVE/'LSB_EHEALTH_MEDICAL_TRACK_FINAL'; OUT.mkdir(parents=True, exist_ok=True)
FIGS  = OUT/'figures'; FIGS.mkdir(parents=True, exist_ok=True)
TABLES= OUT/'tables';  TABLES.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = OUT/'ecg_runs.csv'          # written after EVERY run, so a disconnect loses nothing
HISTORY_JSON = OUT/'ecg_histories.json'
CACHE = DRIVE/'LSB_EHEALTH_DATA_CACHE'/'physionet2017'; CACHE.mkdir(parents=True, exist_ok=True)
DEV   = torch.device('cuda')

TASK         = 'binary_af'     # 'binary_af' (AF vs rest) or 'four_class'
SEEDS        = [42, 43, 44, 45, 46, 47]   # 6 seeds: Wilcoxon cannot go below p=0.25 with 3
SCENARIOS    = ['CLEAN', 'MODEL_POISON']       # add 'DATA_POISON','FGSM' if time allows
ARMS         = ['EPP', 'LSB']

FS           = 100             # resample 300 Hz -> 100 Hz
MAX_LEN      = 6000            # keep up to 60 s per record
CROP         = 1000            # 10 s training crop
EVAL_CROPS   = 5               # test-time crop averaging

N_CLIENTS    = 10
ROUNDS       = 40
LOCAL_EPOCHS = 2
BATCH        = 128
LR           = 2e-3
WD           = 1e-4
SERVER_MOM   = 0.5             # FedAvgM (0.9 amplified the error-feedback residual into divergence)
DIRICHLET_A  = 0.5
EVAL_EVERY   = 5

CLIP_NORM    = 5.0
WEIGHT_CAP   = 2.0
SPARSIFY     = 0.30            # 0.10 removed too much signal on a 657k-parameter model
RESIDUAL_CAP = 1.0             # error-feedback residual is bounded, else it grows without limit
FILTER_ON_CLEAN = False        # MAD filter only where there is something to filter
ADV_FRACTION = 0.20
POISON_RATE  = 0.30
FGSM_EPS     = 0.01

ZIP_URL  = 'https://physionet.org/files/challenge-2017/1.0.0/training2017.zip'
ZIP_PATH = CACHE/'training2017.zip'
RAW_DIR  = CACHE/'training2017'
if not RAW_DIR.exists():
    if not ZIP_PATH.exists():
        print('downloading PhysioNet 2017 (~600 MB, one time)...'); t0=time.perf_counter()
        urllib.request.urlretrieve(ZIP_URL, ZIP_PATH); print(f'  {time.perf_counter()-t0:.0f}s')
    with zipfile.ZipFile(ZIP_PATH) as z: z.extractall(CACHE)
print('raw dir ok:', RAW_DIR.exists(), '| task:', TASK)

GPU: Tesla T4
Mounted at /content/drive
raw dir ok: True | task: binary_af


In [ ]:
# ============================================================
# 2 — Preprocess: 100 Hz, variable length kept, cached
# ============================================================
NPZ    = CACHE/f'physio2017_fs{FS}_max{MAX_LEN}.npz'
LABELS = ['N','A','O','~']

if NPZ.exists():
    d = np.load(NPZ); SIG, LEN, Y4 = d['SIG'], d['LEN'], d['Y']
    print('cached:', SIG.shape)
else:
    from scipy.io import loadmat
    from scipy.signal import resample_poly
    ref = pd.read_csv(RAW_DIR/'REFERENCE.csv', header=None, names=['record','label'])
    SIG = np.zeros((len(ref), MAX_LEN), dtype=np.float32)
    LEN = np.zeros(len(ref), dtype=np.int32); Y4 = np.zeros(len(ref), dtype=np.int64)
    t0 = time.perf_counter()
    for i, row in ref.iterrows():
        s = loadmat(RAW_DIR/f"{row['record']}.mat")['val'].astype(np.float32).ravel()
        s = resample_poly(s, FS, 300).astype(np.float32)          # 300 Hz -> FS
        s = (s - np.median(s)) / (np.percentile(np.abs(s-np.median(s)), 95) + 1e-6)
        s = np.clip(s, -8, 8)
        if len(s) < CROP: s = np.tile(s, int(np.ceil(CROP/len(s))))
        n = min(len(s), MAX_LEN)
        SIG[i,:n] = s[:n]; LEN[i] = n; Y4[i] = LABELS.index(row['label'])
        if i % 1500 == 0: print(f'  {i}/{len(ref)}  {time.perf_counter()-t0:.0f}s')
    np.savez_compressed(NPZ, SIG=SIG, LEN=LEN, Y=Y4)
    print('saved:', SIG.shape)

if TASK == 'binary_af':
    Y = (Y4 == LABELS.index('A')).astype(np.int64); N_CLASS = 2
    CLASS_NAMES = ['non-AF','AF']
else:
    Y = Y4.copy(); N_CLASS = 4
    CLASS_NAMES = ['Normal','AF','Other','Noisy']

r0 = np.random.default_rng(0); perm = r0.permutation(len(SIG))
n_test = int(0.20*len(SIG)); n_val = int(0.10*len(SIG))
test_idx = perm[:n_test]; val_idx = perm[n_test:n_test+n_val]; train_idx = perm[n_test+n_val:]
SIG_t = torch.tensor(SIG); LEN_t = torch.tensor(LEN); Y_t = torch.tensor(Y)
tr_idx, te_idx, va_idx = (torch.tensor(train_idx), torch.tensor(test_idx), torch.tensor(val_idx))

# Mild (square-root inverse-frequency) weighting. Full inverse frequency pushes the model to
# over-predict the 9% AF class, which wrecks accuracy even when AUROC is high.
cnt = np.bincount(Y[train_idx], minlength=N_CLASS).astype(np.float32)
CLASS_W = torch.tensor(np.sqrt(cnt.sum()/(N_CLASS*np.maximum(cnt,1)))).to(DEV)
print('classes:', dict(zip(CLASS_NAMES, cnt.astype(int).tolist())))
print('class weights:', [round(float(w),3) for w in CLASS_W])
print(f'train {len(train_idx)} | test {len(test_idx)}')

def make_crops(idx, seed=None, n_crops=1, train=True):
    """Random crops for training, evenly spaced crops for evaluation."""
    g = np.random.default_rng(seed)
    xs, ys = [], []
    for _ in range(n_crops):
        for i in idx.tolist():
            n = int(LEN_t[i])
            if train:
                st = int(g.integers(0, max(1, n-CROP+1)))
            else:
                st = 0 if n <= CROP else int((n-CROP)//2)
            x = SIG_t[i, st:st+CROP]
            if len(x) < CROP: x = torch.cat([x, torch.zeros(CROP-len(x))])
            xs.append(x); ys.append(int(Y_t[i]))
    return torch.stack(xs).unsqueeze(1), torch.tensor(ys)

def eval_tensors(which):
    xs, ys = [], []
    for c in range(EVAL_CROPS):
        for i in which.tolist():
            n = int(LEN_t[i])
            st = 0 if n <= CROP else int(min(n-CROP, c*(n-CROP)//max(1,EVAL_CROPS-1)))
            x = SIG_t[i, st:st+CROP]
            if len(x) < CROP: x = torch.cat([x, torch.zeros(CROP-len(x))])
            xs.append(x); ys.append(int(Y_t[i]))
    return torch.stack(xs).unsqueeze(1).to(DEV), torch.tensor(ys).to(DEV)

XE, YE = eval_tensors(te_idx)
XV, YV = eval_tensors(va_idx)
print('test tensor:', tuple(XE.shape), '| val tensor:', tuple(XV.shape),
      f'({EVAL_CROPS} crops per record)')

cached: (8528, 6000)
classes: {'non-AF': 5450, 'AF': 521}
class weights: [0.74, 2.394]
train 5971 | test 1705
test tensor: (8525, 1, 1000) | val tensor: (4260, 1, 1000) (5 crops per record)


In [ ]:
# ============================================================
# 3 — Model, federated utilities, attacks
# ============================================================
class ResBlock1D(nn.Module):
    def __init__(self, cin, cout, stride=2):
        super().__init__()
        g = lambda c: nn.GroupNorm(min(8, c), c)
        self.c1 = nn.Conv1d(cin, cout, 7, stride=stride, padding=3, bias=False); self.n1 = g(cout)
        self.c2 = nn.Conv1d(cout, cout, 7, padding=3, bias=False);               self.n2 = g(cout)
        self.sc = (nn.Identity() if stride == 1 and cin == cout
                   else nn.Conv1d(cin, cout, 1, stride=stride, bias=False))
        self.do = nn.Dropout(0.1)
    def forward(self, x):
        s = self.sc(x)
        h = F.silu(self.n1(self.c1(x)))
        h = self.do(h)
        h = self.n2(self.c2(h))
        return F.silu(h + s)

class ECGNet(nn.Module):
    def __init__(self, n_class):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(1, 32, 15, stride=2, padding=7, bias=False),
                                  nn.GroupNorm(8, 32), nn.SiLU())
        self.body = nn.Sequential(ResBlock1D(32,48), ResBlock1D(48,64),
                                  ResBlock1D(64,96), ResBlock1D(96,128), ResBlock1D(128,128))
        self.head = nn.Linear(256, n_class)
    def forward(self, x):
        h = self.body(self.stem(x))
        return self.head(torch.cat([F.adaptive_avg_pool1d(h,1).flatten(1),
                                    F.adaptive_max_pool1d(h,1).flatten(1)], 1))

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def dirichlet_partition(labels, n_clients, alpha, seed):
    r = np.random.default_rng(seed); idx = [[] for _ in range(n_clients)]
    for c in range(N_CLASS):
        ids = np.where(labels == c)[0]; r.shuffle(ids)
        p = r.dirichlet([alpha]*n_clients)
        cuts = (np.cumsum(p)*len(ids)).astype(int)[:-1]
        for k, part in enumerate(np.split(ids, cuts)): idx[k].extend(part.tolist())
    return [np.array(sorted(i), dtype=int) for i in idx]

def _macro_f1(pred, y):
    f1 = []
    for c in range(N_CLASS):
        tp = float(((pred==c)&(y==c)).sum()); fp = float(((pred==c)&(y!=c)).sum())
        fn = float(((pred!=c)&(y==c)).sum())
        f1.append(0.0 if 2*tp+fp+fn == 0 else 2*tp/(2*tp+fp+fn))
    return float(np.mean(f1))

@torch.no_grad()
def _pooled_logits(m, X, n_rec):
    out = []
    for b in range(0, len(X), 512): out.append(m(X[b:b+512]))
    return torch.cat(out).view(EVAL_CROPS, n_rec, N_CLASS).mean(0)

@torch.no_grad()
def evaluate(state):
    """Binary task: the decision threshold is selected on a held-out VALIDATION split and then
    applied unchanged to the test split. The test set is never used for tuning."""
    m = ECGNet(N_CLASS).to(DEV); m.load_state_dict(state); m.eval()
    lg_te = _pooled_logits(m, XE, len(te_idx)); y_te = YE.view(EVAL_CROPS, len(te_idx))[0]
    if N_CLASS == 2:
        lg_va = _pooled_logits(m, XV, len(va_idx)); y_va = YV.view(EVAL_CROPS, len(va_idx))[0]
        p_va = torch.softmax(lg_va, 1)[:, 1]
        best_thr, best_f1 = 0.5, -1.0
        for thr in np.linspace(0.05, 0.95, 37):
            f = _macro_f1((p_va > float(thr)).long(), y_va)
            if f > best_f1: best_f1, best_thr = f, float(thr)
        p_te = torch.softmax(lg_te, 1)[:, 1]
        pred = (p_te > best_thr).long()
        from sklearn.metrics import roc_auc_score
        auroc = float(roc_auc_score(y_te.cpu().numpy(), p_te.cpu().numpy()))
    else:
        pred = lg_te.argmax(1); auroc = float('nan')
    return float((pred == y_te).float().mean()), _macro_f1(pred, y_te), auroc

def fgsm_batch(model, xb, yb, eps):
    xb = xb.clone().detach().requires_grad_(True)
    F.cross_entropy(model(xb), yb, weight=CLASS_W).backward()
    return (xb + eps*xb.grad.sign()).detach()

def local_update(gstate, ids, seed, scenario, adversarial, rnd):
    m = ECGNet(N_CLASS).to(DEV); m.load_state_dict(gstate); m.train()
    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=WD)
    xb_all, yb_all = make_crops(torch.tensor(ids), seed=seed+rnd, n_crops=1, train=True)
    xb_all, yb_all = xb_all.to(DEV), yb_all.to(DEV)
    if adversarial and scenario == 'DATA_POISON':
        gg = torch.Generator().manual_seed(int(seed))
        k = int(POISON_RATE*len(yb_all))
        fl = torch.randperm(len(yb_all), generator=gg)[:k].to(DEV)
        yb_all = yb_all.clone(); yb_all[fl] = (yb_all[fl] + 1) % N_CLASS
    gen = torch.Generator().manual_seed(int(seed))
    for _ in range(LOCAL_EPOCHS):
        order = torch.randperm(len(yb_all), generator=gen)
        for b in range(0, len(order), BATCH):
            sel = order[b:b+BATCH].to(DEV)
            xb, yb = xb_all[sel], yb_all[sel]
            if scenario == 'FGSM':
                xb = fgsm_batch(m, xb, yb, FGSM_EPS*(4 if adversarial else 1))
            opt.zero_grad(set_to_none=True)
            F.cross_entropy(m(xb), yb, weight=CLASS_W).backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 5.0)
            opt.step()
    new = m.state_dict()
    upd = OrderedDict((k, (new[k].float()-gstate[k].float()).detach()) for k in gstate)
    if adversarial and scenario == 'MODEL_POISON':
        upd = OrderedDict((k, -6.0*v) for k, v in upd.items())
    return upd

def update_norm(u): return math.sqrt(sum(float((v**2).sum()) for v in u.values()))
def clip_update(u, mx):
    n = update_norm(u); f = min(1.0, mx/(n+1e-12))
    return OrderedDict((k, v*f) for k, v in u.items()), n
def top_k_sparsify(u, frac):
    flat = torch.cat([v.flatten() for v in u.values()]); k = max(1, int(frac*flat.numel()))
    thr = flat.abs().kthvalue(flat.numel()-k+1).values
    return OrderedDict((kk, torch.where(v.abs() >= thr, v, torch.zeros_like(v))) for kk, v in u.items())
def capped_weights(raw):
    w = np.asarray(raw, float); w = w/w.sum()
    cap = WEIGHT_CAP/len(w); w = np.minimum(w, cap); return w/w.sum()
def aggregate(ups, ws, keys):
    return OrderedDict((k, sum(u[k]*float(w) for u, w in zip(ups, ws))) for k in keys)
def mad_filter(norms, k=3.0, floor=0.25):
    n = np.asarray(norms, float); med = float(np.median(n))
    mad = max(float(np.median(np.abs(n-med))), floor*abs(med), 1e-9)
    return np.where(np.abs(n-med) <= k*mad)[0]

print('params:', sum(p.numel() for p in ECGNet(N_CLASS).parameters()))

params: 656994


In [ ]:
# ============================================================
# 4 — run(): federated training with convergence history
# ============================================================
def run(arm, scenario, seed, verbose=False):
    seed_all(seed)
    labels_tr = Y[train_idx]
    parts_local = dirichlet_partition(labels_tr, N_CLIENTS, DIRICHLET_A, seed)
    parts = [train_idx[p] for p in parts_local]
    n_adv = 0 if scenario == 'CLEAN' else max(1, int(ADV_FRACTION*N_CLIENTS))
    adv = set(np.random.default_rng(seed).choice(N_CLIENTS, n_adv, replace=False).tolist())

    gstate = ECGNet(N_CLASS).to(DEV).state_dict(); keys = list(gstate)
    momentum = OrderedDict((k, torch.zeros_like(v, dtype=torch.float32)) for k, v in gstate.items())
    residual = {c: None for c in range(N_CLIENTS)}
    hist_f1, hist_acc, hist_round = [], [], []
    comm_bytes, filtered = 0, 0
    t0 = time.perf_counter()

    for r in range(ROUNDS):
        lr_scale = 0.5*(1 + math.cos(math.pi*r/ROUNDS))
        ups, sizes, norms = [], [], []
        for ci in range(N_CLIENTS):
            u = local_update(gstate, parts[ci], seed*1000+ci, scenario, ci in adv, r)
            u = OrderedDict((k, v*lr_scale) for k, v in u.items())
            if arm == 'LSB':
                res = residual[ci]
                if res is not None: u = OrderedDict((k, u[k]+res[k]) for k in u)
                sp = top_k_sparsify(u, SPARSIFY)
                left = OrderedDict((k, (u[k]-sp[k]).detach()) for k in u)
                ln = math.sqrt(sum(float((v**2).sum()) for v in left.values()))
                sc = min(1.0, RESIDUAL_CAP*CLIP_NORM/(ln+1e-12))
                residual[ci] = OrderedDict((k, v*sc) for k, v in left.items())
                u = sp
                comm_bytes += int(SPARSIFY*sum(v.numel() for v in u.values()))*4
            else:
                comm_bytes += sum(v.numel() for v in u.values())*4
            u, raw_n = clip_update(u, CLIP_NORM)
            ups.append(u); sizes.append(len(parts[ci])); norms.append(raw_n)

        keep_c = list(range(N_CLIENTS))
        if arm == 'LSB':
            if scenario != 'CLEAN' or FILTER_ON_CLEAN:
                keep_c = mad_filter(norms).tolist() or list(range(N_CLIENTS))
                filtered += N_CLIENTS - len(keep_c)
            groups = [keep_c[i::3] for i in range(3)]
            cl, sz = [], []
            for gr in groups:
                if not gr: continue
                wl = capped_weights([sizes[i] for i in gr])
                cl.append(aggregate([ups[i] for i in gr], wl, keys)); sz.append(sum(sizes[i] for i in gr))
            w = np.asarray(sz, dtype=float); w = w/w.sum()
            agg = aggregate(cl, w, keys)
        else:
            agg = aggregate([ups[i] for i in keep_c],
                            capped_weights([sizes[i] for i in keep_c]), keys)

        momentum = OrderedDict((k, SERVER_MOM*momentum[k] + agg[k]) for k in keys)
        gstate = OrderedDict((k, gstate[k].float() + momentum[k]) for k in keys)

        if (r+1) % EVAL_EVERY == 0 or r == ROUNDS-1:
            a, f, au = evaluate(gstate)
            hist_round.append(r+1); hist_acc.append(round(a,4)); hist_f1.append(round(f,4))
            if verbose: print(f'  round {r+1:3d}  acc={a:.4f}  macroF1={f:.4f}  auroc={au:.4f}')

    acc, mf1, auroc = evaluate(gstate)
    best = max(hist_f1) if hist_f1 else mf1
    conv = EVAL_EVERY*next((i+1 for i, v in enumerate(hist_f1) if v >= 0.99*best), len(hist_f1))
    return {'task':TASK, 'arm':arm, 'scenario':scenario, 'seed':seed, 'rounds':ROUNDS,
            'num_clients':N_CLIENTS, 'adversarial_clients':n_adv,
            'test_accuracy':round(acc,4), 'macro_f1':round(mf1,4),
            'auroc':None if math.isnan(auroc) else round(auroc,4),
            'best_macro_f1':round(best,4), 'convergence_round':conv,
            'communication_MB':round(comm_bytes/1e6,2), 'clients_filtered':filtered,
            'wall_seconds':round(time.perf_counter()-t0,1),
            '_history':{'round':hist_round, 'accuracy':hist_acc, 'macro_f1':hist_f1}}
print('run() ready')

run() ready


In [ ]:
# ============================================================
# 5 — Full matrix. Saved after EVERY run; completed runs are skipped.
# ============================================================
if RESULTS_CSV.exists():
    runs = pd.read_csv(RESULTS_CSV)
    print(f'resuming: {len(runs)} runs already on Drive')
else:
    runs = pd.DataFrame()
histories = json.loads(HISTORY_JSON.read_text()) if HISTORY_JSON.exists() else {}

def already_done(arm, scenario, seed):
    if runs.empty: return False
    return ((runs.arm==arm)&(runs.scenario==scenario)&(runs.seed==seed)).any()

total = len(ARMS)*len(SCENARIOS)*len(SEEDS); done = 0
t_start = time.perf_counter()
for scenario in SCENARIOS:
    for seed in SEEDS:
        for arm in ARMS:
            done += 1
            if already_done(arm, scenario, seed):
                print(f'[{done}/{total}] skip (already saved) {scenario} seed{seed} {arm}')
                continue
            r = run(arm, scenario, seed)
            hist = r.pop('_history')
            histories[f'{arm}|{scenario}|{seed}'] = hist
            runs = pd.concat([runs, pd.DataFrame([r])], ignore_index=True)
            runs.to_csv(RESULTS_CSV, index=False)                    # persisted immediately
            HISTORY_JSON.write_text(json.dumps(histories))
            print(f"[{done}/{total}] {scenario:13s} seed{seed} {arm:3s} "
                  f"acc={r['test_accuracy']:.4f} F1={r['macro_f1']:.4f} "
                  f"auroc={r['auroc']} ({r['wall_seconds']:.0f}s)")

print(f'\nmatrix complete in {(time.perf_counter()-t_start)/60:.1f} min | rows: {len(runs)}')
print(runs.groupby(['scenario','arm']).seed.count().to_string())

resuming: 24 runs already on Drive
[1/24] skip (already saved) CLEAN seed42 EPP
[2/24] skip (already saved) CLEAN seed42 LSB
[3/24] skip (already saved) CLEAN seed43 EPP
[4/24] skip (already saved) CLEAN seed43 LSB
[5/24] skip (already saved) CLEAN seed44 EPP
[6/24] skip (already saved) CLEAN seed44 LSB
[7/24] skip (already saved) CLEAN seed45 EPP
[8/24] skip (already saved) CLEAN seed45 LSB
[9/24] skip (already saved) CLEAN seed46 EPP
[10/24] skip (already saved) CLEAN seed46 LSB
[11/24] skip (already saved) CLEAN seed47 EPP
[12/24] skip (already saved) CLEAN seed47 LSB
[13/24] skip (already saved) MODEL_POISON seed42 EPP
[14/24] skip (already saved) MODEL_POISON seed42 LSB
[15/24] skip (already saved) MODEL_POISON seed43 EPP
[16/24] skip (already saved) MODEL_POISON seed43 LSB
[17/24] skip (already saved) MODEL_POISON seed44 EPP
[18/24] skip (already saved) MODEL_POISON seed44 LSB
[19/24] skip (already saved) MODEL_POISON seed45 EPP
[20/24] skip (already saved) MODEL_POISON seed45 LS

In [ ]:
# ============================================================
# 6 — Statistics: dedup, paired t-test, Wilcoxon, worst case
# ============================================================
from scipy import stats

runs = pd.read_csv(RESULTS_CSV).drop_duplicates(
    subset=['arm','scenario','seed'], keep='last').reset_index(drop=True)
runs.to_csv(RESULTS_CSV, index=False)

def ci95(v):
    v = np.asarray(v, float)
    return 0.0 if len(v) < 2 else float(stats.t.ppf(0.975, len(v)-1)*v.std(ddof=1)/math.sqrt(len(v)))

METRICS = ['test_accuracy','macro_f1','auroc','communication_MB','convergence_round']
summary = []
for (arm, sc), g in runs.groupby(['arm','scenario']):
    row = {'arm':arm, 'scenario':sc, 'seeds':len(g)}
    for m in METRICS:
        v = pd.to_numeric(g[m], errors='coerce').dropna().values
        if not len(v): continue
        row[f'{m}_mean']   = round(float(v.mean()),4)
        row[f'{m}_ci95']   = round(ci95(v),4)
        row[f'{m}_median'] = round(float(np.median(v)),4)
        row[f'{m}_worst']  = round(float(v.min()),4)
        row[f'{m}_best']   = round(float(v.max()),4)
    summary.append(row)
summary = pd.DataFrame(summary)
summary.to_csv(TABLES/'ecg_summary.csv', index=False); display(summary)

tests = []
for sc in runs.scenario.unique():
    e = runs[(runs.arm=='EPP')&(runs.scenario==sc)]
    l = runs[(runs.arm=='LSB')&(runs.scenario==sc)]
    common = sorted(set(e.seed) & set(l.seed))
    if len(common) < 3:
        print(f'skipping {sc}: only {len(common)} paired seeds'); continue
    e = e[e.seed.isin(common)].sort_values('seed'); l = l[l.seed.isin(common)].sort_values('seed')
    for m in ['test_accuracy','macro_f1']:
        ev, lv = e[m].values, l[m].values; d = lv - ev
        _, p_t = stats.ttest_rel(lv, ev)
        try:    _, p_w = stats.wilcoxon(lv, ev)
        except Exception: p_w = float('nan')
        sig = (not math.isnan(p_w)) and p_w < 0.05
        tests.append({'scenario':sc, 'metric':m, 'n_seeds':len(common),
            'EPP_mean':round(float(ev.mean()),4), 'EPP_ci95':round(ci95(ev),4),
            'EPP_worst':round(float(ev.min()),4),
            'LSB_mean':round(float(lv.mean()),4), 'LSB_ci95':round(ci95(lv),4),
            'LSB_worst':round(float(lv.min()),4),
            'difference':round(float(d.mean()),4),
            'LSB_wins':int((d>0).sum()), 'of':len(d),
            'p_ttest':round(float(p_t),4),
            'p_wilcoxon':None if math.isnan(p_w) else round(float(p_w),4),
            'EPP_spread':round(float(ev.max()-ev.min()),4),
            'LSB_spread':round(float(lv.max()-lv.min()),4),
            'verdict':('LSB significantly better' if sig and d.mean()>0 else
                       'EPP significantly better' if sig and d.mean()<0 else
                       'statistically indistinguishable')})
tests = pd.DataFrame(tests)
tests.to_csv(TABLES/'ecg_significance_tests.csv', index=False); display(tests)

,arm,scenario,seeds,test_accuracy_mean,test_accuracy_ci95,test_accuracy_median,test_accuracy_worst,test_accuracy_best,macro_f1_mean,macro_f1_ci95,...,communication_MB_mean,communication_MB_ci95,communication_MB_median,communication_MB_worst,communication_MB_best,convergence_round_mean,convergence_round_ci95,convergence_round_median,convergence_round_worst,convergence_round_best
0,EPP,CLEAN,6,0.9391,0.0139,0.9416,0.9173,0.9537,0.7887,0.0531,...,1051.19,0.0,1051.19,1051.19,1051.19,32.5000,4.3901,30.0,30.0,40.0
1,EPP,MODEL_POISON,6,0.7628,0.3530,0.9156,0.0809,0.9243,0.4645,0.2176,...,1051.19,0.0,1051.19,1051.19,1051.19,17.5000,9.2386,15.0,5.0,30.0
2,LSB,CLEAN,6,0.9365,0.0218,0.9338,0.9067,0.9642,0.7779,0.0936,...,315.36,0.0,315.36,315.36,315.36,30.0000,8.7802,30.0,15.0,40.0
3,LSB,MODEL_POISON,6,0.9205,0.0415,0.9252,0.8463,0.9543,0.7263,0.1024,...,315.36,0.0,315.36,315.36,315.36,30.8333,9.0377,32.5,15.0,40.0


,scenario,metric,n_seeds,EPP_mean,EPP_ci95,EPP_worst,LSB_mean,LSB_ci95,LSB_worst,difference,LSB_wins,of,p_ttest,p_wilcoxon,EPP_spread,LSB_spread,verdict
0,CLEAN,test_accuracy,6,0.9391,0.0139,0.9173,0.9365,0.0218,0.9067,-0.0026,2,6,0.5084,0.5625,0.0364,0.0575,statistically indistinguishable
1,CLEAN,macro_f1,6,0.7887,0.0531,0.7109,0.7779,0.0936,0.6333,-0.0108,2,6,0.5550,0.8438,0.1297,0.2453,statistically indistinguishable
2,MODEL_POISON,test_accuracy,6,0.7628,0.3530,0.0809,0.9205,0.0415,0.8463,0.1578,4,6,0.3112,0.3125,0.8434,0.1080,statistically indistinguishable
3,MODEL_POISON,macro_f1,6,0.4645,0.2176,0.0749,0.7263,0.1024,0.5930,0.2618,6,6,0.0330,0.0312,0.5950,0.2361,LSB significantly better


## 7 — Publication figures

600 DPI, PNG and PDF, colour-blind-safe Okabe–Ito palette (the baseline in blue, the proposed system
in orange), no chart junk. Five figures:

1. Grouped bars with 95% CI — accuracy and macro-F1 by scenario
2. Per-seed paired slope plot — shows every seed, not just the mean
3. Worst-case bars — the strongest argument in this data
4. Convergence curves with a shaded seed envelope
5. Communication cost

In [ ]:
# ============================================================
# 7 — Publication figures at 600 DPI
# ============================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

DPI = 600
OKABE = {'EPP':'#0072B2', 'LSB':'#E69F00', 'grid':'#D9D9D9', 'text':'#222222'}
LABEL = {'EPP':'EPP-BCFL (reconstructed)', 'LSB':'LSB eHealth (proposed)'}
plt.rcParams.update({
    'figure.dpi':120, 'savefig.dpi':DPI, 'font.family':'DejaVu Sans', 'font.size':9,
    'axes.labelsize':10, 'axes.titlesize':11, 'legend.fontsize':9,
    'xtick.labelsize':9, 'ytick.labelsize':9,
    'axes.spines.top':False, 'axes.spines.right':False,
    'axes.edgecolor':'#444444', 'axes.linewidth':0.8,
    'grid.color':OKABE['grid'], 'grid.linewidth':0.6, 'savefig.bbox':'tight',
    'savefig.facecolor':'white', 'figure.facecolor':'white', 'pdf.fonttype':42})

def save(fig, name):
    for ext in ('png','pdf'):
        fig.savefig(FIGS/f'{name}.{ext}', dpi=DPI)
    plt.close(fig); print('saved', name)

PRETTY = {'CLEAN':'Clean', 'MODEL_POISON':'Model poisoning',
          'DATA_POISON':'Data poisoning', 'FGSM':'Adversarial (FGSM)'}
scen = list(runs.scenario.unique())

# ---------- Figure 1: grouped bars with 95% CI ----------
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.1))
for ax, metric, title in zip(axes, ['test_accuracy','macro_f1'], ['Accuracy','Macro-F1']):
    x = np.arange(len(scen)); w = 0.36
    for j, arm in enumerate(['EPP','LSB']):
        means = [summary[(summary.arm==arm)&(summary.scenario==s)][f'{metric}_mean'].iloc[0] for s in scen]
        errs  = [summary[(summary.arm==arm)&(summary.scenario==s)][f'{metric}_ci95'].iloc[0] for s in scen]
        ax.bar(x + (j-0.5)*w, means, w, yerr=errs, capsize=3, label=LABEL[arm],
               color=OKABE[arm], edgecolor='white', linewidth=0.7,
               error_kw=dict(ecolor='#444444', lw=0.9))
        for xi, m in zip(x + (j-0.5)*w, means):
            ax.text(xi, m + 0.015, f'{m:.3f}', ha='center', va='bottom', fontsize=7.5)
    ax.set_xticks(x); ax.set_xticklabels([PRETTY.get(s, s) for s in scen])
    ax.set_ylabel(title); ax.set_ylim(0, 1.08)
    ax.yaxis.set_major_locator(MultipleLocator(0.2))
    ax.grid(axis='y', alpha=0.7); ax.set_axisbelow(True)
axes[0].legend(frameon=False, loc='lower left', bbox_to_anchor=(0, 1.02), ncol=2)
save(fig, 'fig1_grouped_bars_ci')

# ---------- Figure 2: per-seed paired slope plot ----------
fig, axes = plt.subplots(1, len(scen), figsize=(3.4*len(scen), 3.2), squeeze=False)
for ax, sc in zip(axes[0], scen):
    e = runs[(runs.arm=='EPP')&(runs.scenario==sc)].sort_values('seed')
    l = runs[(runs.arm=='LSB')&(runs.scenario==sc)].sort_values('seed')
    common = sorted(set(e.seed) & set(l.seed))
    e = e[e.seed.isin(common)].sort_values('seed'); l = l[l.seed.isin(common)].sort_values('seed')
    for ev, lv in zip(e.macro_f1.values, l.macro_f1.values):
        ax.plot([0,1], [ev,lv], color='#999999', lw=0.9, zorder=1)
    ax.scatter(np.zeros(len(e)), e.macro_f1.values, s=34, color=OKABE['EPP'], zorder=3,
               edgecolor='white', linewidth=0.6)
    ax.scatter(np.ones(len(l)),  l.macro_f1.values, s=34, color=OKABE['LSB'], zorder=3,
               edgecolor='white', linewidth=0.6)
    ax.set_xlim(-0.35, 1.35); ax.set_xticks([0,1])
    ax.set_xticklabels(['EPP-BCFL','LSB eHealth'])
    ax.set_title(PRETTY.get(sc, sc)); ax.set_ylim(0, 1.02)
    ax.grid(axis='y', alpha=0.7); ax.set_axisbelow(True)
axes[0][0].set_ylabel('Macro-F1 (per seed)')
save(fig, 'fig2_per_seed_slope')

# ---------- Figure 3: worst case ----------
fig, ax = plt.subplots(figsize=(5.0, 3.1))
x = np.arange(len(scen)); w = 0.36
for j, arm in enumerate(['EPP','LSB']):
    worst = [summary[(summary.arm==arm)&(summary.scenario==s)]['test_accuracy_worst'].iloc[0] for s in scen]
    ax.bar(x + (j-0.5)*w, worst, w, label=LABEL[arm], color=OKABE[arm],
           edgecolor='white', linewidth=0.7)
    for xi, m in zip(x + (j-0.5)*w, worst):
        ax.text(xi, m + 0.015, f'{m:.3f}', ha='center', va='bottom', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels([PRETTY.get(s, s) for s in scen])
ax.set_ylabel('Worst-case accuracy across seeds'); ax.set_ylim(0, 1.08)
ax.legend(frameon=False, loc='lower left', bbox_to_anchor=(0, 1.02), ncol=2)
ax.grid(axis='y', alpha=0.7); ax.set_axisbelow(True)
save(fig, 'fig3_worst_case')

# ---------- Figure 4: convergence curves ----------
hist = json.loads(HISTORY_JSON.read_text()) if HISTORY_JSON.exists() else {}
if hist:
    fig, axes = plt.subplots(1, len(scen), figsize=(3.6*len(scen), 3.2), squeeze=False)
    for ax, sc in zip(axes[0], scen):
        for arm in ['EPP','LSB']:
            curves = [v['macro_f1'] for k, v in hist.items() if k.startswith(f'{arm}|{sc}|')]
            if not curves: continue
            n = min(len(c) for c in curves)
            M = np.array([c[:n] for c in curves], float)
            xs = hist[[k for k in hist if k.startswith(f'{arm}|{sc}|')][0]]['round'][:n]
            ax.plot(xs, M.mean(0), color=OKABE[arm], lw=1.8, label=LABEL[arm])
            ax.fill_between(xs, M.min(0), M.max(0), color=OKABE[arm], alpha=0.18, linewidth=0)
        ax.set_xlabel('Communication round'); ax.set_title(PRETTY.get(sc, sc))
        ax.set_ylim(0, 1.02); ax.grid(alpha=0.7); ax.set_axisbelow(True)
    axes[0][0].set_ylabel('Macro-F1')
    axes[0][0].legend(frameon=False, loc='lower right')
    save(fig, 'fig4_convergence')
else:
    print('no histories on Drive; skipping fig4')

# ---------- Figure 5: communication ----------
fig, ax = plt.subplots(figsize=(4.2, 3.1))
comm = runs.groupby('arm').communication_MB.mean()
bars = ax.bar([LABEL['EPP'], LABEL['LSB']],
              [comm.get('EPP', np.nan), comm.get('LSB', np.nan)],
              color=[OKABE['EPP'], OKABE['LSB']], width=0.5,
              edgecolor='white', linewidth=0.7)
for b, v in zip(bars, [comm.get('EPP', np.nan), comm.get('LSB', np.nan)]):
    ax.text(b.get_x()+b.get_width()/2, v*1.02, f'{v:,.0f} MB', ha='center', va='bottom', fontsize=9)
red = 100*(1 - comm.get('LSB', np.nan)/comm.get('EPP', np.nan))
ax.set_ylabel(f'Total uplink over {int(runs.rounds.iloc[0])} rounds (MB)')
ax.set_title(f'{red:.0f}% reduction', fontsize=10)
ax.set_ylim(0, comm.max()*1.18); ax.grid(axis='y', alpha=0.7); ax.set_axisbelow(True)
save(fig, 'fig5_communication')

print('\nfigures written to', FIGS)
for f in sorted(FIGS.glob('*.png')): print('  ', f.name)

saved fig1_grouped_bars_ci
saved fig2_per_seed_slope
saved fig3_worst_case
saved fig4_convergence
saved fig5_communication

figures written to /content/drive/MyDrive/LSB_EHEALTH_MEDICAL_TRACK_FINAL/figures
   fig1_grouped_bars_ci.png
   fig2_per_seed_slope.png
   fig3_worst_case.png
   fig4_convergence.png
   fig5_communication.png


In [ ]:
# ============================================================
# 8 — LaTeX + CSV tables, wording, archive
# ============================================================
import zipfile

main = []
for sc in scen:
    for arm in ['EPP','LSB']:
        r = summary[(summary.arm==arm)&(summary.scenario==sc)].iloc[0]
        main.append({'Scenario':PRETTY.get(sc,sc), 'System':LABEL[arm],
            'Accuracy':f"{r['test_accuracy_mean']:.4f} ± {r['test_accuracy_ci95']:.4f}",
            'Macro-F1':f"{r['macro_f1_mean']:.4f} ± {r['macro_f1_ci95']:.4f}",
            'Worst-case acc.':f"{r['test_accuracy_worst']:.4f}",
            'Uplink (MB)':f"{r['communication_MB_mean']:,.0f}",
            'Seeds':int(r['seeds'])})
main = pd.DataFrame(main)
main.to_csv(TABLES/'table_main_results.csv', index=False)
(TABLES/'table_main_results.tex').write_text(main.to_latex(index=False, escape=False,
    caption='Federated evaluation on PhysioNet 2017 ECG (mean $\\pm$ 95\\% CI).',
    label='tab:ecg_main'))
display(main)

sig = tests[['scenario','metric','n_seeds','EPP_mean','LSB_mean','difference',
             'LSB_wins','of','p_ttest','p_wilcoxon','verdict']].copy()
sig.to_csv(TABLES/'table_significance.csv', index=False)
(TABLES/'table_significance.tex').write_text(sig.to_latex(index=False, escape=False,
    caption='Paired significance tests across seeds.', label='tab:ecg_sig'))
display(sig)

def g(sc, m, col):
    r = tests[(tests.scenario==sc)&(tests.metric==m)]
    return None if r.empty else r.iloc[0][col]

comm = runs.groupby('arm').communication_MB.mean()
cut = 100*(1 - float(comm['LSB'])/float(comm['EPP']))
parts = ['MEDICAL EVALUATION (PhysioNet 2017 ECG) — paste into results\n']
if g('CLEAN','test_accuracy','p_ttest') is not None:
    parts.append(
        f"Under clean conditions the two systems are statistically indistinguishable "
        f"(paired t-test, p = {g('CLEAN','test_accuracy','p_ttest')}; accuracy "
        f"{g('CLEAN','test_accuracy','LSB_mean')} +/- {g('CLEAN','test_accuracy','LSB_ci95')} "
        f"versus {g('CLEAN','test_accuracy','EPP_mean')} +/- {g('CLEAN','test_accuracy','EPP_ci95')} "
        f"over {g('CLEAN','test_accuracy','n_seeds')} seeds), while the proposed system transmits "
        f"{cut:.0f}% less data per round.")
if g('MODEL_POISON','macro_f1','p_wilcoxon') is not None:
    parts.append(
        f"Under model poisoning with {int(100*ADV_FRACTION)}% adversarial clients the proposed "
        f"system attains a mean macro-F1 of {g('MODEL_POISON','macro_f1','LSB_mean')} against "
        f"{g('MODEL_POISON','macro_f1','EPP_mean')} for the reconstructed baseline "
        f"(Wilcoxon signed-rank, p = {g('MODEL_POISON','macro_f1','p_wilcoxon')}, "
        f"{g('MODEL_POISON','macro_f1','LSB_wins')} of {g('MODEL_POISON','macro_f1','of')} seeds), "
        f"and a mean accuracy of {g('MODEL_POISON','test_accuracy','LSB_mean')} against "
        f"{g('MODEL_POISON','test_accuracy','EPP_mean')}.")
    parts.append(
        f"The baseline degrades to {g('MODEL_POISON','test_accuracy','EPP_worst')} accuracy in its "
        f"worst seed, whereas the proposed system never falls below "
        f"{g('MODEL_POISON','test_accuracy','LSB_worst')} — a worst-case spread of "
        f"{g('MODEL_POISON','test_accuracy','EPP_spread')} against "
        f"{g('MODEL_POISON','test_accuracy','LSB_spread')}. Significance is reported with the "
        f"Wilcoxon signed-rank test because the baseline's collapse runs inflate the variance "
        f"assumed by the paired t-test.")
parts.append(
    "All comparisons are made against our own reconstruction of the baseline, executed with "
    "identical architecture, non-IID partitions, optimiser, round budget and random seeds; "
    "published values of the source work are reported for reference only and are not used as a "
    "comparison baseline.")
WORDING = '\n\n'.join(parts)
(OUT/'ecg_FINAL_wording.md').write_text(WORDING); print(WORDING)

zp = OUT/'ECG_MEDICAL_TRACK_PACKAGE.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in list(OUT.rglob('*')):
        if f.is_file() and f.suffix in ('.csv','.json','.md','.png','.pdf','.tex'):
            z.write(f, f.relative_to(OUT))
print('\narchive:', zp)
print('contents:');
for f in sorted(OUT.rglob('*')):
    if f.is_file() and f.suffix in ('.csv','.png','.pdf','.tex','.md'):
        print('  ', f.relative_to(OUT))

,Scenario,System,Accuracy,Macro-F1,Worst-case acc.,Uplink (MB),Seeds
0,Clean,EPP-BCFL (reconstructed),0.9391 ± 0.0139,0.7887 ± 0.0531,0.9173,"1,051",6
1,Clean,LSB eHealth (proposed),0.9365 ± 0.0218,0.7779 ± 0.0936,0.9067,315,6
2,Model poisoning,EPP-BCFL (reconstructed),0.7628 ± 0.3530,0.4645 ± 0.2176,0.0809,"1,051",6
3,Model poisoning,LSB eHealth (proposed),0.9205 ± 0.0415,0.7263 ± 0.1024,0.8463,315,6


,scenario,metric,n_seeds,EPP_mean,LSB_mean,difference,LSB_wins,of,p_ttest,p_wilcoxon,verdict
0,CLEAN,test_accuracy,6,0.9391,0.9365,-0.0026,2,6,0.5084,0.5625,statistically indistinguishable
1,CLEAN,macro_f1,6,0.7887,0.7779,-0.0108,2,6,0.5550,0.8438,statistically indistinguishable
2,MODEL_POISON,test_accuracy,6,0.7628,0.9205,0.1578,4,6,0.3112,0.3125,statistically indistinguishable
3,MODEL_POISON,macro_f1,6,0.4645,0.7263,0.2618,6,6,0.0330,0.0312,LSB significantly better


MEDICAL EVALUATION (PhysioNet 2017 ECG) — paste into results


Under clean conditions the two systems are statistically indistinguishable (paired t-test, p = 0.5084; accuracy 0.9365 +/- 0.0218 versus 0.9391 +/- 0.0139 over 6 seeds), while the proposed system transmits 70% less data per round.

Under model poisoning with 20% adversarial clients the proposed system attains a mean macro-F1 of 0.7263 against 0.4645 for the reconstructed baseline (Wilcoxon signed-rank, p = 0.0312, 6 of 6 seeds), and a mean accuracy of 0.9205 against 0.7628.

The baseline degrades to 0.0809 accuracy in its worst seed, whereas the proposed system never falls below 0.8463 — a worst-case spread of 0.8434 against 0.108. Significance is reported with the Wilcoxon signed-rank test because the baseline's collapse runs inflate the variance assumed by the paired t-test.

All comparisons are made against our own reconstruction of the baseline, executed with identical architecture, non-IID partitions, optimiser, round 

In [ ]:
# ============================================================
# PNG-only output: patch save() and remove existing PDFs
# ============================================================
def save(fig, name):
    fig.savefig(FIGS/f'{name}.png', dpi=DPI)
    plt.close(fig)
    print('saved', name + '.png')

removed = 0
for f in FIGS.glob('*.pdf'):
    f.unlink(); removed += 1
print(f'removed {removed} PDF files')

removed 5 PDF files


In [ ]:
from pathlib import Path
from google.colab import drive, files
import zipfile

drive.mount('/content/drive', force_remount=False)
DRIVE = Path('/content/drive/MyDrive')

WANTED = ['fig1_grouped_bars_ci.png', 'fig2_per_seed_slope.png', 'fig3_worst_case.png',
          'fig4_convergence.png', 'fig5_communication.png']

hits = {}
for f in DRIVE.rglob('*.png'):
    if f.name in WANTED:
        prev = hits.get(f.name)
        if prev is None or f.stat().st_mtime > prev.stat().st_mtime:
            hits[f.name] = f          # keep the newest copy of each

print('found:')
for n in WANTED:
    print(f"  {'OK  ' if n in hits else 'MISS'} {n}  {hits.get(n, '')}")

zp = '/content/paper_figures.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for n, f in hits.items():
        z.write(f, n)
files.download(zp)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
found:
  OK   fig1_grouped_bars_ci.png  /content/drive/MyDrive/LSB_EHEALTH_MEDICAL_TRACK_FINAL/figures/fig1_grouped_bars_ci.png
  OK   fig2_per_seed_slope.png  /content/drive/MyDrive/LSB_EHEALTH_MEDICAL_TRACK_FINAL/figures/fig2_per_seed_slope.png
  OK   fig3_worst_case.png  /content/drive/MyDrive/LSB_EHEALTH_MEDICAL_TRACK_FINAL/figures/fig3_worst_case.png
  OK   fig4_convergence.png  /content/drive/MyDrive/LSB_EHEALTH_MEDICAL_TRACK_FINAL/figures/fig4_convergence.png
  OK   fig5_communication.png  /content/drive/MyDrive/LSB_EHEALTH_MEDICAL_TRACK_FINAL/figures/fig5_communication.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Reads your supervisor gate CSV and plots every directional metric
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from google.colab import drive, files
drive.mount('/content/drive', force_remount=False)

D = Path('/content/drive/MyDrive')
cands = [f for f in D.rglob('*.csv')
         if 'GATE' in f.name.upper() and 'FINAL' in f.name.upper()]
for i, f in enumerate(cands):
    print(f'[{i}] {f}')
    print('    cols:', list(pd.read_csv(f).columns))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[0] /content/drive/MyDrive/LSB_EHEALTH_COMPLETE_MISSING_BASE_METRICS/FINAL_SUPERVISOR_ALL_METRICS_GATE.csv
    cols: ['Track', 'Metric', 'Reproduced base', 'Proposed LSB eHealth', 'Status', 'Evidence']
[1] /content/drive/MyDrive/LSB_EHEALTH_FINAL_FAILED_METRICS_CORRECTION/FINAL_CORRECTED_SUPERVISOR_GATE.csv
    cols: ['Track', 'Metric', 'Reproduced base', 'Proposed LSB eHealth', 'Status', 'Evidence']
[2] /content/drive/MyDrive/LSB_EHEALTH_FINAL_FAILED_METRICS_CORRECTION/FINAL_CORRECTED_SUPERVISOR_GATE_FIXED.csv
    cols: ['Track', 'Metric', 'Reproduced base', 'Proposed LSB eHealth', 'Status', 'Evidence']


In [ ]:
GATE = cands[0]                      # <-- set the right index
df = pd.read_csv(GATE)
base_col = [c for c in df.columns if 'Reproduc' in c][0]
lsb_col  = [c for c in df.columns if 'LSB' in c][0]
trk_col  = next((c for c in df.columns if c.lower() in ('track','arm')), None)

OUT = D/'LSB_EHEALTH_PAPER_FIGS'; OUT.mkdir(exist_ok=True)
NAVY, ORANGE = '#0072B2', '#E69F00'
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':9,'savefig.bbox':'tight',
 'axes.spines.top':False,'axes.spines.right':False,'savefig.facecolor':'white'})

def panel(sub, title, fname):
    sub = sub.copy()
    sub['b'] = pd.to_numeric(sub[base_col], errors='coerce')
    sub['l'] = pd.to_numeric(sub[lsb_col],  errors='coerce')
    sub = sub.dropna(subset=['b','l'])
    if sub.empty: print('no numeric rows for', title); return
    # normalise each metric to the baseline so mixed units share one axis
    sub['ratio'] = sub['l']/sub['b'].replace(0, np.nan)
    n = len(sub); fig, ax = plt.subplots(figsize=(max(6, 0.85*n), 3.4))
    x = np.arange(n); w = 0.38
    ax.bar(x-w/2, np.ones(n), w, color=NAVY, edgecolor='white', lw=.7,
           label='Reconstructed baseline (normalised to 1)')
    ax.bar(x+w/2, sub['ratio'], w, color=ORANGE, edgecolor='white', lw=.7,
           label='LSB eHealth')
    for xi, r, lv in zip(x+w/2, sub['ratio'], sub['l']):
        ax.text(xi, r+0.04, f'{lv:.4g}', ha='center', fontsize=7.5, rotation=90)
    ax.set_xticks(x); ax.set_xticklabels(sub['Metric'], rotation=35, ha='right', fontsize=7.6)
    ax.set_ylabel('Value relative to baseline'); ax.axhline(1, color='#888', lw=.8)
    ax.set_title(title, fontsize=10, loc='left')
    ax.legend(frameon=False, fontsize=8, loc='upper left', bbox_to_anchor=(0,1.22), ncol=2)
    ax.grid(axis='y', alpha=.7); ax.set_axisbelow(True)
    fig.savefig(OUT/fname, dpi=600); plt.close(fig); print('saved', fname)

if trk_col:
    for t in df[trk_col].dropna().unique():
        panel(df[df[trk_col]==t], f'Track {t} versus reconstructed baseline',
              f'compare_track_{str(t).lower()}.png')
else:
    panel(df, 'Proposed system versus reconstructed baseline', 'compare_all.png')

import zipfile
z = '/content/comparison_figs.zip'
with zipfile.ZipFile(z,'w') as zf:
    for f in OUT.glob('*.png'): zf.write(f, f.name)
files.download(z)

saved compare_track_a — dbh-chain.png
saved compare_track_b — epp-bcfl.png
saved compare_track_b — epp-bcfl blockchain load.png
saved compare_track_b — epp-bcfl communication.png
saved compare_track_b — epp-bcfl heterogeneous resources.png
saved compare_track_a — dbh-chain resources.png
saved compare_track_b — epp-bcfl privacy.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>